# Trénování modelu pro rozpoznávání gest ruky

**Projekt:** Rozpoznávání gest ruky pomocí Machine Learning  
**Autor:** Student maturitního projektu  

---

## Co tento notebook dělá?

Tento notebook zpracovává data nasbíraná z webkamery (souřadnice kloubů ruky) a z nich trénuje model, který umí rozpoznat gesto ruky.

Postup je typický **ML pipeline**:

1. Načtení dat z CSV souboru
2. Čištění dat
3. Převod textových názvů gest na čísla
4. Rozdělení na trénovací a testovací sadu
5. Škálování (normalizace) hodnot
6. Trénování modelu
7. Vyhodnocení přesnosti
8. Uložení modelu na disk

---

### Použité knihovny

| Knihovna | K čemu slouží |
|---|---|
| `pandas` | Načítání a práce s tabulkovými daty |
| `scikit-learn` | ML algoritmy, předzpracování, vyhodnocení |
| `matplotlib` + `seaborn` | Vizualizace výsledků |
| `joblib` | Ukládání natrénovaného modelu |

## Krok 1 – Načtení dat

Nejprve musíme data dostat do paměti programu. Nasbíraná data jsou uložena v souboru `dataset.csv`.

Používáme knihovnu **pandas**, která umí CSV soubory načíst do tzv. **DataFrame** – to je tabulka, se kterou se v Pythonu pracuje velmi snadno.

**Proč Pandas?** Pandas je standardní nástroj pro práci s tabulkovými daty v Pythonu. Nabízí hotové funkce pro načítání, filtrování i transformaci dat, takže nemusíme psát vlastní parser CSV souboru.

Na konci si zobrazíme prvních 5 řádků, abychom vizuálně ověřili, že data vypadají správně (správný počet sloupců, žádné divné hodnoty).

In [ ]:
# Importujeme knihovnu pandas a zkrátíme její název na "pd" – to je všeobecně
# uznávaná zkratka, používají ji všichni Python programátoři.
import pandas as pd

# Načteme CSV soubor do DataFrame (tabulky).
# Pandas automaticky rozpozná záhlaví (první řádek = názvy sloupců).
data = pd.read_csv("dataset.csv")

# Vypíšeme počet řádků a sloupců, abychom věděli, kolik dat máme.
print("Počet řádků (snímků):", data.shape[0])
print("Počet sloupců:", data.shape[1])

# Zobrazíme prvních 5 řádků tabulky pro vizuální kontrolu.
data.head()

## Krok 2 – Čištění dat

Reálná data jsou zřídkakdy dokonalá. Mohou obsahovat **chybějící hodnoty (NaN)** – to nastane například tehdy, když MediaPipe v daném snímku ruku nenašel, ale řádek se přesto zapsal prázdný.

**Proč je NaN problém?** Matematické operace s prázdnou hodnotou nejdou provést. Většina ML algoritmů by při výskytu NaN okamžitě spadla s chybou.

**Řešení:** Jednoduše odstraníme všechny řádky, kde se NaN nachází. Protože máme stovky snímků na každé gesto, ztráta několika neúplných řádků výsledek nijak neovlivní.

In [ ]:
# Zjistíme, kolik řádků obsahuje alespoň jednu chybějící hodnotu (NaN).
pocet_nan = data.isnull().sum().sum()
print("Počet chybějících hodnot před čištěním:", pocet_nan)

# Odstraníme všechny řádky, které obsahují NaN kdekoliv.
# inplace=True znamená, že upravenou tabulku uložíme přímo zpět do proměnné "data"
# (nechceme vytvářet novou kopii tabulky).
data.dropna(inplace=True)

# Obnovíme indexování řádků od 0, protože po smazání některých řádků
# by číslování mohlo mít "díry" (např. 0, 1, 5, 6...).
# drop=True znamená, že starý index zahodíme a nepřidáme ho jako nový sloupec.
data.reset_index(drop=True, inplace=True)

# Ověříme, kolik dat nám zbylo po čištění.
print("Počet řádků po čištění:", data.shape[0])

## Krok 3 – Transformace dat: převod textových labelů na čísla

Sloupec `label` obsahuje textové řetězce jako `"stop"`, `"ok"`, `"mír"`. Počítače (a ML algoritmy) ale pracují pouze s čísly.

Použijeme **LabelEncoder** ze scikit-learn, který každému unikátnímu textu přiřadí číslo:
- `"mír"` → `0`
- `"ok"` → `1`  
- `"stop"` → `2`
- ... (abecedně seřazeno)

**Proč ukládáme encoder?** LabelEncoder si zapamatuje, které číslo patří ke kterému textu. Budeme ho potřebovat i při predikci v reálném programu – aby model mohl vrátit slovní název gesta místo čísla.

In [ ]:
# Importujeme LabelEncoder ze scikit-learn.
from sklearn.preprocessing import LabelEncoder

# Vytvoříme instanci LabelEncoder – prázdný "překladač" bez dat.
le = LabelEncoder()

# fit_transform() udělá dvě věci najednou:
#   1. "fit" = encoder si projde všechny unikátní hodnoty ve sloupci "label"
#              a přiřadí každé z nich číslo (interně si to zapamatuje).
#   2. "transform" = nahradí textové hodnoty odpovídajícími čísly.
# Výsledek uložíme zpět do sloupce "label".
data["label"] = le.fit_transform(data["label"])

# Vypíšeme, jak encoder přeložil texty na čísla (pro dokumentaci a kontrolu).
print("Mapování gest na čísla:")
for cislo, text in enumerate(le.classes_):
    print(" ", cislo, "→", text)

## Krok 4 – Rozdělení dat: příznaky vs. cíl, trénovací vs. testovací sada

### 4a) Rozdělení na X a y

Data musíme rozdělit do dvou skupin:
- **X (příznaky / features):** Vše, co model dostane jako vstup = souřadnice kloubů ruky (63 číselných sloupců).
- **y (cíl / target):** Co chceme, aby model předpověděl = číslo gesta (sloupec `label`).

### 4b) Trénovací a testovací sada (poměr 80/20)

**Proč rozdělujeme data?** Kdybychom model trénovali na všech datech a pak ho testovali na stejných datech, mohlo by dojít k tzv. **overfittingu** – model by si data "nazpaměť" a zdánlivě by měl 100% přesnost, ale na nových datech by selhal.

Správný postup je:
- **80 % dat** = trénovací sada → z těchto dat se model učí
- **20 % dat** = testovací sada → tato data model nikdy neviděl; slouží ke spravedlivému ověření přesnosti

Parametr `random_state=42` zajistí, že rozdělení je vždy stejné (pro reprodukovatelnost výsledků).

In [ ]:
# Importujeme funkci pro rozdělení dat ze scikit-learn.
from sklearn.model_selection import train_test_split

# X = všechny sloupce KROMĚ "label" (to jsou vstupní příznaky).
# data.drop() vytvoří kopii tabulky bez zadaného sloupce.
# axis=1 znamená, že odstraňujeme sloupec (ne řádek).
X = data.drop("label", axis=1)

# y = pouze sloupec "label" (to je cílová hodnota, kterou chceme předpovídat).
y = data["label"]

# Rozdělíme X a y na trénovací a testovací část.
# test_size=0.2 → 20 % dat půjde do testovací sady, 80 % do trénovací.
# random_state=42 → pevné "náhodné" semínko pro reprodukovatelné výsledky.
#                   Číslo 42 je konvenční hodnota; záleží jen na tom, aby bylo pevné.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Ověříme velikosti výsledných sad.
print("Trénovací sada (X):", X_train.shape)
print("Testovací sada (X): ", X_test.shape)

## Krok 5 – Škálování dat (StandardScaler)

Toto je jeden z nejdůležitějších kroků předzpracování dat.

**Proč škálovat?** Naše souřadnice x, y jsou v rozsahu 0.0–1.0, ale souřadnice z (hloubka) mohou být záporné i velice malé (např. -0.05). Jiné modely pracují s hodnotami v tisících. Bez škálování by algoritmus mohl klást větší důraz na hodnoty s větší číselnou hodnotou – ne proto, že jsou důležitější, ale jen proto, že jsou větší čísla.

**StandardScaler** přetransformuje každý sloupec tak, aby měl:
- **průměr = 0**
- **směrodatnou odchylku = 1**

**Klíčové pravidlo:**
- `fit_transform()` používáme **pouze na trénovací data** – scaler se "naučí" průměr a odchylku z trénovacích dat.
- `transform()` (bez fit) používáme na testovací data – stejné parametry se jen aplikují, scaler se z testovacích dat nic nesmí učit. Jinak bychom "podváděli" a výsledky by nebyly reálné.

In [ ]:
# Importujeme StandardScaler ze scikit-learn.
from sklearn.preprocessing import StandardScaler

# Vytvoříme instanci scaleru – prázdný objekt, který ještě nic neví.
scaler = StandardScaler()

# fit_transform na TRÉNOVACÍCH datech:
#   - "fit" = scaler spočítá průměr a směrodatnou odchylku pro každý sloupec
#             (naučí se z trénovacích dat, jak data "vypadají")
#   - "transform" = přepočítá hodnoty na standardizované (průměr 0, odchylka 1)
X_train = scaler.fit_transform(X_train)

# transform (BEZ fit) na TESTOVACÍCH datech:
# Používáme STEJNÉ parametry (průměr, odchylku), které se scaler naučil
# z trénovacích dat. Testovací data musí být transformována stejným způsobem,
# ale scaler se z nich NIC NESMÍ učit – jinak bychom prozradili modelu info
# z testovací sady a výsledky by nebyly důvěryhodné.
X_test = scaler.transform(X_test)

print("Škálování dokončeno.")
print("Průměr prvního příznaku po škálování (trénovací):", X_train[:, 0].mean().round(6))

## Krok 6 – Trénování modelu: Random Forest Classifier

Jako klasifikační algoritmus jsme zvolili **Random Forest** (náhodný les).

**Jak funguje?**  
Random Forest se skládá z mnoha rozhodovacích stromů (v našem případě 100). Každý strom se učí na trochu jiné podmnožině dat. Při predikci každý strom "hlasuje" pro jedno gesto a vyhrává to, pro které hlasovalo nejvíce stromů.

**Proč Random Forest?**
- Funguje skvěle i bez rozsáhlého ladění parametrů
- Je odolný vůči přeučení (overfitting)
- Nevyžaduje škálování dat (my jsme ho přesto provedli – je to dobrá praxe a nutné, pokud bychom model v budoucnu zaměnili za SVM nebo neuronovou síť)
- Podává výborné výsledky na tabulkových datech

Parametr `n_estimators=100` říká, že les bude tvořen 100 stromy.

In [ ]:
# Importujeme RandomForestClassifier ze scikit-learn.
from sklearn.ensemble import RandomForestClassifier

# Vytvoříme model (zatím prázdný, nenaučený).
#   n_estimators=100  → les bude složen ze 100 rozhodovacích stromů
#   random_state=42   → pevné semínko pro reprodukovatelné výsledky
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Spustíme trénování – model se naučí z trénovacích dat.
# X_train = vstupní příznaky (souřadnice kloubů) trénovací sady
# y_train = správné odpovědi (čísla gest) trénovací sady
print("Trénuji model, čekej prosím ...")
model.fit(X_train, y_train)

print("Trénování dokončeno!")

## Krok 7 – Vyhodnocení modelu

### Accuracy (přesnost)
Nejjednodušší metrika. Říká: „Z celkového počtu testovacích snímků, kolik procent jsme správně rozpoznali?"  
Například `accuracy = 0.97` znamená 97% přesnost.

### Matice záměn (Confusion Matrix)
Accuracy nám neřekne, *kde* model chybuje. Matice záměn je tabulka, kde:
- **Řádky** = skutečná gesta (co opravdu bylo)
- **Sloupce** = předpovězená gesta (co model říkal)
- **Diagonála** = správné předpovědi (čím světlejší/větší, tím lépe)
- **Mimo diagonálu** = chyby (model si jedno gesto spletl s jiným)

Matice záměn nám umožní odhalit, například, zda si model plete gesta „ok" a „mír" – a podle toho nasbírat více dat pro ta problematická gesta.

In [ ]:
# Importujeme potřebné nástroje pro vyhodnocení a vizualizaci.
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Necháme model předpovědět gesta pro testovací data.
# model.predict() vezme souřadnice kloubů a vrátí číslo předpovězeného gesta.
y_pred = model.predict(X_test)

# Spočítáme přesnost: porovnáme předpovědi (y_pred) se správnými odpověďmi (y_test).
presnost = accuracy_score(y_test, y_pred)
print("Přesnost modelu na testovacích datech: {:.2f} %".format(presnost * 100))

# Vytvoříme matici záměn – tabulku chyb a správných předpovědí.
# Řádky = skutečná gesta, sloupce = předpovězená gesta.
matice = confusion_matrix(y_test, y_pred)

# Nastavíme velikost obrázku (šířka 8 palců, výška 6 palců).
plt.figure(figsize=(8, 6))

# Vykreslíme matici záměn jako barevnou tabulku (heatmap).
# annot=True    → zobrazí čísla uvnitř buněk
# fmt="d"       → čísla jako celá čísla (integer), ne desetinná
# cmap="Blues"  → barevná škála v modrých odstínech
# xticklabels a yticklabels → nahradí čísla zpět textovými názvy gest
sns.heatmap(
    matice,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=le.classes_,
    yticklabels=le.classes_
)

# Přidáme popisky os a nadpis grafu.
plt.xlabel("Předpovězené gesto")
plt.ylabel("Skutečné gesto")
plt.title("Matice záměn (Confusion Matrix)")

# Zobrazíme graf.
plt.tight_layout()
plt.show()

## Krok 8 – Export modelu

Natrénovaný model a scaler musíme uložit na disk, abychom je mohli použít v reálném programu (v `collect_data.py` nebo v novém predikčním skriptu) bez nutnosti model znovu trénovat.

Ukládáme **dva soubory**:

| Soubor | Co obsahuje | Proč ho potřebujeme |
|---|---|---|
| `model.pkl` | Natrénovaný RandomForest | Provádí samotnou predikci gesta |
| `scaler.pkl` | Natrénovaný StandardScaler | Musíme škálovat vstup STEJNÝM způsobem jako při trénování |

**Proč ukládat i scaler?** Kdyby při predikci vstoupila nová souřadnice `x = 0.75` a my bychom použili jiný (nový) scaler, výsledek by byl jiný než při trénování. Model by pak dostával jinak přeškálovaná čísla a předpovědi by byly špatné.

Knihovna **joblib** je součástí scikit-learn a je optimalizovaná na ukládání velkých numpy polí (jako je RandomForest).

In [ ]:
# Importujeme knihovnu joblib, která umí ukládat Python objekty na disk.
import joblib

# Uložíme natrénovaný model do souboru "model.pkl".
# Přípona .pkl (pickle) je konvence pro uložené Python objekty.
joblib.dump(model, "model.pkl")
print("Model uložen do souboru: model.pkl")

# Uložíme scaler do souboru "scaler.pkl".
# Při predikci v reálném programu musíme nová data škálovat
# STEJNÝM scalerem jako při trénování – proto ho ukládáme.
joblib.dump(scaler, "scaler.pkl")
print("Scaler uložen do souboru: scaler.pkl")

# Uložíme také LabelEncoder, abychom při predikci mohli převést
# číslo zpět na textový název gesta (např. 0 → "mír").
joblib.dump(le, "label_encoder.pkl")
print("LabelEncoder uložen do souboru: label_encoder.pkl")

print()
print("=== HOTOVO ===")
print("Soubory model.pkl, scaler.pkl a label_encoder.pkl jsou připraveny.")
print("Nahraj je do stejné složky jako predikční skript.")